# BRAIN × FINN_SC: fire emissions in Santa Catarina in 2019, on the same grid and the same region

This notebook does three things:

1. **Downloads** the 365 daily fire-emission files of BRAIN (FINN v1.5 processed for CMAQ, 4 km grid of
   southern Brazil) listed in `BRAIN_output_urls.txt`.
2. **Extracts** from them the emissions of Santa Catarina (SC) in the units, species, days and region of the
   FINN_SC inventory, after checking on every file how it was built.
3. **Compares** BRAIN with the FINN_SC records of 2019 (FINN v2.5 MODIS and FINN v2.5 MODIS+VIIRS): annual
   totals, months, days, grid cells and municipalities. Optional: the same check against the FINN v1.5 input
   files, to separate a version effect from a processing effect.

It answers the co-author's suggestion to compare the inventory with other inventories. BRAIN covers 2019 only,
so the comparison is for 2019.

**Run it with** `Runtime ▸ Run all`. Every result is cached on Google Drive, so a second run (or a run after a
disconnection) continues where it stopped.

## How the BRAIN fire files were made (and what this notebook checks)

BRAIN uses FINN v1.5 and converts it to hourly 3D CMAQ input with `finn2cmaq` (Hoinaski et al., 2024, section
2.1.3). The rules below were read in the finn2cmaq source code (commit `88bb2f6`, 4 Aug 2022, whose species list
is Table S3 of Hoinaski et al., 2024) and in PseudoNetCDF, the library finn2cmaq uses for the grid.
**The notebook does not assume them: section 3 tests each one on the real files and stops if one fails.**

| Step | What finn2cmaq does | Where it is written | Checked in |
|---|---|---|---|
| Input | FINN v1.5 MODIS, GEOS-Chem speciation. The BRAIN file names follow the output template of `run_nrt.sh` (`FINNv1.5_2016.CB6r3.NRT.<grid>.3D.<date>`), the script that processes the near-real-time daily files `GLOB_GEOSchem_YYYYJJJ.txt.gz`; "2016" is a fixed part of that template | `run_nrt.sh` | 9 (optional) |
| Grid | each fire point goes to the 4 km cell that contains it: Mercator (IOAPI `GDTYP` 7) on a sphere of 6 370 000 m, index by truncation; fires with `AREA` < 50 m² are dropped | `txt2daily.py`; PseudoNetCDF `ll2ij` | 0, 3 |
| Day | the FINN total of day *d* goes into the file of date *d* as 25 hourly rates (00 UTC of *d* to 00 UTC of *d*+1) shaped by a local-time profile (profile 210 by default before Sep 2022) | `daily2hourly3d.py`, `tpro.txt` | 3, 5 |
| Layers | the default `layerfrac.csv` spreads each fire over 19 layers whose fractions add up to **1.01** | `layerfrac.csv` | 5 |
| Species | CO, NO, NO₂, SO₂, NH₃ and `ECH4` (= FINN CH₄) in mol s⁻¹; `POC` = OC, `PEC` = BC, `PSO4` = 1.26 % of PM₂.₅, `PNO3` = 0.13 % of PM₂.₅, `PMOTHR` = the rest of PM₂.₅, in g s⁻¹ | `gc12_to_cb6r3_ae7.txt` | 3, 5 |

Three consequences for the comparison follow from the code:

* **The daily total of a BRAIN file is the sum of its first 24 hourly rates × 3600 s.** With profile 210 these
  24 rates carry exactly 100 % of the FINN day (the 25th rate belongs to the next file). The day of a BRAIN file
  is therefore the FINN `DAY`, the same day field the FINN_SC records use; no time-zone shift is needed.
* **BRAIN totals are 1.01 × the FINN input** if the default layer fractions were used (checked in section 5).
* **CO₂, total NMOC (COV) and PM₁₀ cannot be compared**: CO₂ and PM₁₀ are not written, and NMOC is split into
  lumped CB6 species whose moles cannot be turned back into NMOC mass. PM₂.₅ is recovered exactly from `PSO4`.
  Table S3 of Hoinaski et al. (2024) labels `ECH4` as ethane; in the finn2cmaq source `ECH4 = CH4`.

## Comparison design

| Item | Choice | Why |
|---|---|---|
| Datasets | BRAIN (FINN v1.5 MODIS); FINN v2.5 MODIS and FINN v2.5 MODIS+VIIRS from `finn_sc_records_2003_2025.csv.gz` (the MODIS+VIIRS series is the one analysed in the article and report) | v1.5 → v2.5 MODIS shows the version effect; v2.5 MODIS → v2.5 MODIS+VIIRS shows the sensor effect |
| Grid | the FINN_SC points are put on the BRAIN grid with the same function finn2cmaq used | identical cells |
| Region R1 | cells lying entirely inside SC | the same region for every dataset, with no assumption |
| Region R2 | the whole State: BRAIN cells weighted by the fraction of their area inside SC (bounds: interior cells only, and every cell touching SC); FINN_SC records whose point lies in SC | BRAIN border cells also hold fires of Paraná, Rio Grande do Sul and Argentina |
| Days | only days with a valid BRAIN file, for every dataset | equal coverage |
| Units | tonnes; NOₓ as NO mass, as the FINN `NOXasNO` field | same basis as the inventory |
| Statistics | ratios; Pearson and Spearman correlation of monthly totals (n = 12, descriptive); Spearman correlation of daily totals with a moving-block bootstrap 95 % interval (blocks of 7 days, Künsch, 1989), because daily fire series are serially correlated; Spearman correlation over cells and municipalities; share of mass located where the other dataset also has fire | rank statistics suit zero-inflated, skewed emissions |

## Before you run (5 minutes)

1. Put the **FINN_SC** folder in your Google Drive (default `MyDrive/FINN_SC`), or leave it out and the notebook
   tries `git clone`, then asks for the zip.
2. Put **`BRAIN_output_urls.txt`** in `MyDrive/BRAIN_FINN_2019/` (or upload it when asked).
3. Optional: if some BRAIN zips do not download, download them in the browser and drop them, unchanged, in
   `MyDrive/BRAIN_FINN_2019/manual_zips/`. The notebook uses them first.
4. `Runtime ▸ Run all`. For a quick trial, set `ONLY_DATES` in the settings cell to two or three dates first.

Everything the notebook writes goes to `MyDrive/BRAIN_FINN_2019/` (`tables/`, `figures/`, `logs/`,
`cache_days/`).

## 0. Settings

In [ ]:
# ==== SETTINGS: edit here only ===============================================
YEAR = 2019
DRIVE_DIR      = '/content/drive/MyDrive/BRAIN_FINN_2019'   # all outputs and the day cache go here
URLS_FILE      = f'{DRIVE_DIR}/BRAIN_output_urls.txt'        # the URL list (uploaded if missing)
FINN_SC_DIR    = '/content/drive/MyDrive/FINN_SC'             # FINN_SC repository (folder with config/config.yml)
FINN_SC_GIT    = 'https://github.com/jessicajcss/FINN_SC.git' # tried only if FINN_SC_DIR is missing
MANUAL_ZIP_DIR = f'{DRIVE_DIR}/manual_zips'                   # BRAIN zips downloaded by hand (optional)
ZIP_DIR        = '/content/brain_zips'                        # download cache on the Colab disk
NC_TMP         = '/content/brain_nc_tmp'                      # one unzipped netCDF at a time
FINN15_DIR     = '/content/finn15'                            # FINN v1.5 files of section 9 (Colab disk)
KEEP_ZIPS_ON_DRIVE = False   # True also copies every zip to DRIVE_DIR/zips (several GB)
PAUSE_S = 1.0                # pause between downloads (be gentle with the server)

ONLY_DATES = None            # e.g. ['2019-08-20', '2019-09-14'] for a trial; None = every date in the list
EARTH_RADIUS_M = 6370000.0   # PseudoNetCDF / finn2cmaq default; section 0 checks it against PseudoNetCDF
INTERIOR_MIN_FRACTION = 0.999  # a cell is 'inside SC' (region R1) when >= 99.9 % of its area is in SC
CROP_MARGIN_CELLS = 3        # cells read around SC
BLOCK_DAYS, N_BOOT, SEED = 7, 2000, 20261101

RUN_V15_CHECK = True   # section 9a: FINN v1.5 2019 annual file from NSF NCAR (large download)
RUN_NRT_CHECK = True   # section 9b: FINN v1.5 near-real-time daily files (the BRAIN input), if still online
NRT_N_DAYS = 5         # days tested in 9b (the days with most BRAIN CO)

## 1. Setup

Mounts Google Drive, installs the libraries and writes `brain_core.py` (the functions used below; open the
cell to read them, there is no need to edit it).

In [ ]:
import os, sys, subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'netCDF4', 'pyproj', 'geopandas',
                    'shapely>=2', 'pyyaml', 'tqdm', 'requests'], check=True)
for d in [DRIVE_DIR, MANUAL_ZIP_DIR, ZIP_DIR, NC_TMP]:
    os.makedirs(d, exist_ok=True)
print('Colab' if IN_COLAB else 'local run', '| outputs in', DRIVE_DIR)

In [ ]:
%%writefile brain_core.py
"""brain_core.py - BRAIN (FINN v1.5 via finn2cmaq) vs FINN_SC records, same grid and region.

Every rule that reproduces how the BRAIN fire files were built comes from the code
that built them, read on 2026-11-01:
  * finn2cmaq (B. Henderson, https://github.com/barronh/finn2cmaq), commit 88bb2f6
    (2022-08-04): scripts/txt2daily.py, scripts/daily2hourly3d.py, aux/tpro.txt,
    aux/layerfrac.csv, aux/gc12_to_cb6r3_ae7.txt, run_nrt.sh. The species list of that
    commit is the list of Table S3 of Hoinaski et al. (2024, ESSD 16, 2385-2405).
  * PseudoNetCDF 3.5.0 and 3.2.2 (ll2ij, ij2ll, IOAPI map definition), the library
    finn2cmaq uses to put each FINN fire on the CMAQ grid.
Grid of the BRAIN d02 domain (SC_2019): attributes of the BRAIN MEGAN file
MEGANv31.SC_2019.CB6X.2020015.BDSNP.ncf (same dataset, same grid).
"""
from __future__ import annotations

import gzip
import hashlib
import io
import itertools
import json
import os
import re
import shutil
import tarfile
import time
import urllib.parse as up
import zipfile
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd

__version__ = "1.0.0"

# ---------------------------------------------------------------------------
# 0. Constants copied from the sources named above
# ---------------------------------------------------------------------------
EXPECTED_GRID = dict(GDTYP=7, P_ALP=0.0, P_BET=0.0, P_GAM=-54.856998443603516,
                     XCENT=-54.856998443603516, YCENT=0.0, XORIG=-105999.8125,
                     YORIG=-3507558.25, XCELL=4000.0, YCELL=4000.0, NCOLS=233, NROWS=243)
GRID_KEYS = list(EXPECTED_GRID)

# PseudoNetCDF uses IOAPI_ISPH when set, otherwise 6370000 m ("consistent with WRF").
IOAPI_SPHERE_RADIUS = 6370000.0

# finn2cmaq aux/tpro.txt: hour1..hour24 in LOCAL time; finn2cmaq always uses the FIRST row.
# Before 2022-09-16 the first row was profile 210; afterwards it is 1008.
TPRO = {
    210: [0.00533091933213] * 7 + [0.020116676725, 0.04023335345, 0.0704083685375,
          0.100583383625, 0.130758398713, 0.1609334138, 0.170991752163, 0.12070006035,
          0.0704083685375, 0.04023335345] + [0.00533091933213] * 7,
    1008: [0.6, 0.6, 0.6, 0.6, 0.6, 0.65, 0.75, 0.9, 1.1, 1.25, 1.45, 1.6, 1.8, 1.75, 1.7,
           1.55, 1.35, 1.1, 0.9, 0.75, 0.65, 0.6, 0.6, 0.6],
    500: [0.0] * 8 + [0.0416208104052, 0.0416208104052, 0.0833416708354, 0.0833416708354,
          0.125062531266, 0.166683341671, 0.125062531266, 0.0833416708354, 0.0833416708354,
          0.0833416708354, 0.0416208104052, 0.0416208104052] + [0.0] * 4,
}
# finn2cmaq aux/layerfrac.csv (first 20 rows; identical in every commit). The fractions sum to 1.01.
DEFAULT_LAYERFRAC = [0.125, 0.125] + [0.05] * 8 + [0.04] * 9 + [0.0]

# finn2cmaq aux/gc12_to_cb6r3_ae7.txt: PM speciation (fractions of FINN PM25, per kg -> g).
SPECIATION = {
    "finn2cmaq <= 2022-08-04 (BRAIN Table S3)": dict(
        f_so4=0.0126, f_no3=0.0013, marker=None,
        pm_vars=["POC", "PEC", "PSO4", "PNO3", "PMOTHR"]),
    "finn2cmaq >= 2022-08-05 (SPECIATE 91102)": dict(
        f_so4=0.0126015, f_no3=0.001323, marker="PNCOM",
        pm_vars=["POC", "PNCOM", "PEC", "PSO4", "PNO3", "PNH4", "PCL", "PAL", "PCA", "PFE",
                 "PMG", "PMN", "PK", "PSI", "PNA", "PTI", "PMOTHR"]),
}
GAS_VARS = ["CO", "NO", "NO2", "SO2", "NH3", "ECH4"]   # moles/s ; ECH4 = FINN CH4 in finn2cmaq
BASE_PM_VARS = ["POC", "PEC", "PSO4", "PNO3", "PMOTHR"]  # g/s

# Species compared and the column of the FINN_SC records (tonnes) that matches each one.
SPECIES = ["CO", "NOx", "SO2", "NH3", "CH4", "PM25", "OC", "BC"]
OURS_COLUMN = dict(CO="CO", NOx="NOXasNO", SO2="SO2", NH3="NH3", CH4="CH4",
                   PM25="PM25", OC="OC", BC="BC")
SPECIES_LABEL = dict(CO="CO", NOx="NOx (as NO)", SO2="SO2", NH3="NH3", CH4="CH4",
                     PM25="PM2.5", OC="OC", BC="BC")
# Fallback molar masses (g/mol) = FINN_SC config/config.yml (conversion.molar_mass_g_per_mol).
DEFAULT_MOLAR_MASS = {"CO": 28.01, "NO": 30.01, "NO2": 46.01, "SO2": 64.07, "NH3": 17.03, "CH4": 16.04}

# FINN v1.5 GEOS-Chem column order used by FINN_SC (python/finn_sc/download.py).
FINN_V15_COLNAMES = [
    "DAY", "TIME", "GENVEG", "LATI", "LONGI", "AREA", "CO2", "CO", "NO", "NO2",
    "SO2", "NH3", "CH4", "ACET", "ALD2", "ALK4", "BENZ", "C2H2", "C2H4", "C2H6",
    "C3H8", "CH2O", "GLYC", "GLYX", "HAC", "MEK", "MGLY", "PRPE", "TOLU", "XYLE",
    "OC", "BC", "PM25",
]
FINN15_ANNUAL_URLS = ["https://www.acom.ucar.edu/Data/fire/data/finn1/FINNv1.5_{year}.GEOSCHEM.tar.gz",
                      "https://www.acom.ucar.edu/Data/fire/data/finn1/FINNv1.5_{year}.GEOSCHEM.txt.gz"]
# finn2cmaq scripts/get_nrt.py
FINN15_NRT_URL = "https://www.acom.ucar.edu/acresp/MODELING/finn_emis_txt/GLOB_GEOSchem_{yj}.txt.gz"
FINN2CMAQ_AREA_MIN = 50.0   # txt2daily drops fires with AREA < 50 m2 ("assumed false detect")


def md5(path, block=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(block), b""):
            h.update(chunk)
    return h.hexdigest()


def _py(v):
    """numpy / bytes attribute -> plain JSON-friendly Python."""
    if isinstance(v, bytes):
        return v.decode("latin1").strip()
    if isinstance(v, str):
        return v.strip()
    if isinstance(v, np.ndarray):
        return [_py(x) for x in v.tolist()]
    if isinstance(v, (np.integer,)):
        return int(v)
    if isinstance(v, (np.floating,)):
        return float(v)
    return v


# ---------------------------------------------------------------------------
# 1. Grid: replica of PseudoNetCDF (GDTYP 7) as called by finn2cmaq
# ---------------------------------------------------------------------------
@dataclass
class Grid:
    GDTYP: int
    P_ALP: float
    P_BET: float
    P_GAM: float
    XCENT: float
    YCENT: float
    XORIG: float
    YORIG: float
    XCELL: float
    YCELL: float
    NCOLS: int
    NROWS: int
    R: float = IOAPI_SPHERE_RADIUS

    @classmethod
    def from_attrs(cls, attrs, R=IOAPI_SPHERE_RADIUS):
        kw = {k: _py(attrs[k]) for k in GRID_KEYS}
        kw["GDTYP"], kw["NCOLS"], kw["NROWS"] = int(kw["GDTYP"]), int(kw["NCOLS"]), int(kw["NROWS"])
        return cls(R=R, **kw)

    def _proj(self):
        import pyproj
        if self.GDTYP != 7:
            raise NotImplementedError("only IOAPI GDTYP 7 (equatorial Mercator) is implemented")
        # PseudoNetCDF asserts these two equalities for GDTYP 7 before building the projection.
        if not (self.P_ALP == self.YCENT and self.P_GAM == self.XCENT):
            raise ValueError("GDTYP 7 requires P_ALP == YCENT and P_GAM == XCENT (PseudoNetCDF rule)")
        # PseudoNetCDF: +proj=merc (+lat_0=YCENT) +lon_0=XCENT +a=+b=R (+lat_ts defaults to 0)
        return pyproj.Proj(f"+proj=merc +lat_ts=0 +lon_0={self.XCENT!r} +x_0=0 +y_0=0 "
                           f"+a={self.R!r} +b={self.R!r} +units=m +no_defs")

    def ll2index(self, lon, lat):
        """lon/lat (deg) -> continuous grid coordinates (0 at the SW corner of cell 0)."""
        p = self._proj()
        x, y = p(np.asarray(lon, dtype="f8"), np.asarray(lat, dtype="f8"))
        return (np.asarray(x) - self.XORIG) / self.XCELL, (np.asarray(y) - self.YORIG) / self.YCELL

    def ll2ij(self, lon, lat):
        """PseudoNetCDF ll2ij(clean='mask'): truncation toward zero, then out-of-domain mask."""
        xg, yg = self.ll2index(lon, lat)
        i = np.asarray(xg).astype("i")
        j = np.asarray(yg).astype("i")
        inside = (i >= 0) & (i < self.NCOLS) & (j >= 0) & (j < self.NROWS)
        return i, j, inside

    def ij2ll(self, i, j):
        """Cell-centre lon/lat (PseudoNetCDF ij2ll uses i + 0.5, j + 0.5)."""
        p = self._proj()
        x = (np.asarray(i, dtype="f8") + 0.5) * self.XCELL + self.XORIG
        y = (np.asarray(j, dtype="f8") + 0.5) * self.YCELL + self.YORIG
        return p(x, y, inverse=True)


def compare_grid(attrs, expected=EXPECTED_GRID, rtol=1e-7):
    """List of (key, found, expected) for every grid attribute that differs."""
    bad = []
    for k, ev in expected.items():
        fv = _py(attrs.get(k)) if k in attrs else None
        if fv is None or not np.isclose(float(fv), float(ev), rtol=rtol, atol=1e-6):
            bad.append((k, fv, ev))
    return bad


# ---------------------------------------------------------------------------
# 2. finn2cmaq time and layer allocation (for diagnostics)
# ---------------------------------------------------------------------------
def tfactor25(hours24):
    """finn2cmaq gettemporal before the time-zone roll: 25 instantaneous hourly factors."""
    t = np.asarray(hours24, dtype="f8")
    t = t / t.sum()
    tt = np.append(np.append(t[-1], t), t[0])
    xc = np.arange(tt.size)
    return np.interp(xc[:-1] + 0.5, xc, tt)


def tz_offset(lon):
    """finn2cmaq: (lon / 15).round(0) (numpy rounds half to even)."""
    return np.round(np.asarray(lon, dtype="f8") / 15.0, 0).astype(int)


def hourly_factors(profile_id, tzoff):
    return np.roll(tfactor25(TPRO[profile_id]), -int(tzoff))


def match_time_profile(vec25, tzoff):
    """Compare a cell's 25 hourly values (shape only) with every finn2cmaq profile."""
    v = np.asarray(vec25, dtype="f8")
    if v.size != 25 or v.sum() <= 0:
        return dict(best=None, max_abs_diff={})
    v = v / v.sum()
    diffs = {}
    for pid in TPRO:
        e = hourly_factors(pid, tzoff)
        diffs[pid] = float(np.abs(v - e / e.sum()).max())
    best = min(diffs, key=diffs.get)
    return dict(best=best, max_abs_diff=diffs, matched=diffs[best] < 1e-4,
                S24=float(hourly_factors(best, tzoff)[:24].sum()))


def match_layers(vecz):
    """Compare a cell's layer distribution (shape only) with the default layerfrac.csv."""
    v = np.asarray(vecz, dtype="f8")
    if v.sum() <= 0:
        return dict(matched=None)
    nz = v.size
    ref = np.zeros(nz)
    n = min(nz, len(DEFAULT_LAYERFRAC))
    ref[:n] = DEFAULT_LAYERFRAC[:n]
    d = float(np.abs(v / v.sum() - ref / ref.sum()).max())
    return dict(nz=nz, max_abs_diff=d, matched=d < 1e-4, default_sum=float(ref.sum()))


# ---------------------------------------------------------------------------
# 3. URL list, download, unzip
# ---------------------------------------------------------------------------
def parse_brain_urls(path, folder="/V1/FINN/"):
    rows = []
    with open(path, encoding="utf-8", errors="replace") as fh:
        for line in fh:
            u = line.strip()
            if not u.lower().startswith("http"):
                continue
            q = up.parse_qs(up.urlparse(u).query)
            p = q.get("path", [""])[0]
            if folder not in p:
                continue
            fn = q.get("fileName", [os.path.basename(p)])[0]
            m = re.search(r"(\d{4}-\d{2}-\d{2})\.zip$", fn)
            if not m:
                raise ValueError(f"no date in file name: {fn}")
            rows.append(dict(date=pd.Timestamp(m.group(1)), fileName=fn, path=p,
                             fileId=q.get("fileId", [""])[0], url=u))
    df = pd.DataFrame(rows).sort_values("date").reset_index(drop=True)
    dup = df[df.date.duplicated(keep=False)]
    if len(dup):
        raise ValueError(f"duplicated dates in the URL list:\n{dup}")
    return df


def valid_zip(path):
    if not os.path.exists(path) or os.path.getsize(path) < 22:
        return False
    try:
        with zipfile.ZipFile(path) as z:
            return z.testzip() is None
    except zipfile.BadZipFile:
        return False


def download(url, dest, session=None, retries=4, timeout=(30, 600), pause=5.0, check=valid_zip):
    """Stream url -> dest with retries; `check(dest)` must accept the result."""
    import requests
    if check(dest):
        return dest
    s = session or requests.Session()
    s.headers.setdefault("User-Agent", "Mozilla/5.0 (FINN_SC BRAIN comparison notebook)")
    os.makedirs(os.path.dirname(dest) or ".", exist_ok=True)
    last = None
    for attempt in range(1, retries + 1):
        tmp = dest + ".part"
        try:
            with s.get(url, stream=True, timeout=timeout, allow_redirects=True) as r:
                r.raise_for_status()
                with open(tmp, "wb") as fh:
                    for chunk in r.iter_content(1 << 20):
                        fh.write(chunk)
            if not check(tmp):
                with open(tmp, "rb") as fh:
                    head = fh.read(160)
                raise IOError(f"downloaded file failed the format check; first bytes: {head!r}")
            os.replace(tmp, dest)
            return dest
        except Exception as exc:  # noqa: BLE001 - report and retry
            last = exc
            if os.path.exists(tmp):
                os.remove(tmp)
            time.sleep(pause * attempt)
    raise RuntimeError(f"download failed after {retries} attempts: {url}\n  last error: {last}")


def is_gzip(path):
    try:
        with open(path, "rb") as fh:
            return fh.read(2) == b"\x1f\x8b"
    except OSError:
        return False


def extract_nc(zip_path, workdir):
    """Extract the single netCDF member of a BRAIN zip (largest member if several)."""
    os.makedirs(workdir, exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        members = [m for m in z.infolist() if not m.is_dir()]
        if not members:
            raise IOError(f"empty zip: {zip_path}")
        m = max(members, key=lambda x: x.file_size)
        out = os.path.join(workdir, os.path.basename(m.filename))
        with z.open(m) as src, open(out, "wb") as dst:
            shutil.copyfileobj(src, dst, 1 << 20)
    return out, [x.filename for x in members]


# ---------------------------------------------------------------------------
# 4. Reading one BRAIN daily file
# ---------------------------------------------------------------------------
def read_header(nc_path):
    import netCDF4
    with netCDF4.Dataset(nc_path) as ds:
        attrs = {k: _py(ds.getncattr(k)) for k in ds.ncattrs()}
        dims = {k: len(v) for k, v in ds.dimensions.items()}
        variables = {}
        for name, v in ds.variables.items():
            variables[name] = dict(dims=list(v.dimensions), shape=list(v.shape), dtype=str(v.dtype),
                                   units=_py(getattr(v, "units", "")),
                                   long_name=_py(getattr(v, "long_name", "")),
                                   var_desc=_py(getattr(v, "var_desc", "")))
        tflag = np.asarray(ds.variables["TFLAG"][:, 0, :]) if "TFLAG" in ds.variables else None
        fmt = ds.data_model
    return dict(attrs=attrs, dims=dims, variables=variables, data_model=fmt,
                tflag=None if tflag is None else tflag.tolist())


def detect_speciation(variables):
    names = set(variables)
    for label, sp in SPECIATION.items():
        if sp["marker"] and sp["marker"] in names:
            return label
    return "finn2cmaq <= 2022-08-04 (BRAIN Table S3)"


def needed_vars(speciation_label):
    sp = SPECIATION[speciation_label]
    return GAS_VARS + [v for v in sp["pm_vars"] if v not in GAS_VARS]


def check_units(variables, names):
    """finn2cmaq writes 'moles/s' for gases and 'g/s' for aerosols."""
    problems = []
    for n in names:
        if n not in variables:
            problems.append((n, "missing"))
            continue
        u = variables[n]["units"].replace(" ", "").lower()
        want = "moles/s" if n in GAS_VARS else "g/s"
        if u != want:
            problems.append((n, variables[n]["units"]))
    return problems


def extract_day(nc_path, date, crop, varnames, expected_grid=EXPECTED_GRID):
    """Daily amounts per cell (moles/day or g/day) from one BRAIN daily file.

    The daily amount is the sum of the 24 hourly rates stamped with the file's own date
    (TFLAG hours 00..23) times 3600 s, summed over all layers. finn2cmaq writes the whole
    FINN daily total of DAY d into the file of date d (25 steps, the 25th is d+1 00Z), so
    the result is aligned with the FINN DAY of the records.
    Returns (cells DataFrame with date,i,j,<vars>; diagnostics dict).
    """
    import netCDF4
    j0, j1, i0, i1 = crop
    date = pd.Timestamp(date)
    d0 = int(date.strftime("%Y%j"))
    d1 = int((date + pd.Timedelta(days=1)).strftime("%Y%j"))
    diag = dict(date=str(date.date()), file=os.path.basename(nc_path))
    with netCDF4.Dataset(nc_path) as ds:
        ds.set_auto_maskandscale(False)
        attrs = {k: ds.getncattr(k) for k in ds.ncattrs()}
        diag["grid_mismatch"] = compare_grid(attrs, expected_grid)
        diag["SDATE"], diag["TSTEP"] = int(attrs.get("SDATE", -1)), int(attrs.get("TSTEP", -1))
        tf = np.asarray(ds.variables["TFLAG"][:, 0, :])
        steps = np.where(tf[:, 0] == d0)[0]
        hours_ok = len(steps) == 24 and np.array_equal(tf[steps, 1], np.arange(24) * 10000)
        diag["n_steps"] = int(tf.shape[0])
        diag["tflag_ok"] = bool(hours_ok and diag["SDATE"] == d0 and diag["TSTEP"] == 10000)
        diag["has_next_00z"] = bool(tf.shape[0] >= 25 and tf[24, 0] == d1 and tf[24, 1] == 0)
        if not hours_ok:
            raise ValueError(f"{nc_path}: TFLAG does not hold hours 00-23 of {date.date()}")
        out, diag["missing_vars"], diag["negatives"], diag["nonfinite"] = {}, [], {}, {}
        for name in varnames:
            if name not in ds.variables:
                diag["missing_vars"].append(name)
                continue
            v = ds.variables[name]
            if list(v.dimensions) != ["TSTEP", "LAY", "ROW", "COL"]:
                raise ValueError(f"{name}: unexpected dimensions {v.dimensions}")
            a = np.asarray(v[:, :, j0:j1, i0:i1], dtype="f8")
            bad = ~np.isfinite(a) | (np.abs(a) >= 9.0e36)
            if bad.any():
                diag["nonfinite"][name] = int(bad.sum())
                a[bad] = 0.0
            neg = a < 0
            if neg.any():
                diag["negatives"][name] = [int(neg.sum()), float(a[neg].sum() * 3600.0)]
            day = a[steps].sum(axis=(0, 1)) * 3600.0
            out[name] = day
            if name == "CO":
                diag["nlay"] = int(a.shape[1])
                diag["CO_day_mol"] = float(day.sum())
                diag["CO_all25_mol"] = float(a.sum() * 3600.0)
                if day.max() > 0:
                    jj, ii = np.unravel_index(int(np.argmax(day)), day.shape)
                    diag["maxcell_ij"] = [int(ii + i0), int(jj + j0)]
                    diag["maxcell_hourly25"] = a[:, :, jj, ii].sum(axis=1).tolist()
                    diag["maxcell_layers"] = a[:, :, jj, ii].sum(axis=0).tolist()
    if not out:
        raise ValueError(f"{nc_path}: none of the requested variables is present")
    stack = np.stack([out[k] for k in out])
    nz = np.any(stack != 0, axis=0)
    jj, ii = np.nonzero(nz)
    cells = pd.DataFrame({"date": date, "i": ii + i0, "j": jj + j0})
    for k in out:
        cells[k] = out[k][jj, ii]
    return cells, diag


def brain_to_tonnes(cells, molar_mass, speciation_label):
    """BRAIN daily amounts (moles, g) -> tonnes of the compared species.

    NOx (as NO) = (NO + NO2) moles x M(NO), the same basis as the FINN NOXasNO field.
    PM2.5 = FINN PM25 recovered exactly from PSO4 (finn2cmaq: PSO4 = PM25 x f_SO4 x 1000).
    PM25_species_sum = sum of every primary PM variable written by finn2cmaq.
    """
    mm = dict(DEFAULT_MOLAR_MASS)
    mm.update(molar_mass or {})
    sp = SPECIATION[speciation_label]
    out = cells[["date", "i", "j"]].copy()
    out["CO"] = cells["CO"] * mm["CO"] / 1e6
    out["NOx"] = (cells["NO"] + cells["NO2"]) * mm["NO"] / 1e6
    out["SO2"] = cells["SO2"] * mm["SO2"] / 1e6
    out["NH3"] = cells["NH3"] * mm["NH3"] / 1e6
    out["CH4"] = cells["ECH4"] * mm["CH4"] / 1e6
    out["OC"] = cells["POC"] / 1e6
    out["BC"] = cells["PEC"] / 1e6
    out["PM25"] = cells["PSO4"] / sp["f_so4"] / 1e6
    present = [v for v in sp["pm_vars"] if v in cells]
    out["PM25_species_sum"] = cells[present].sum(axis=1) / 1e6
    return out


def speciation_checks(cells, speciation_label):
    """Two identities of the finn2cmaq PM speciation, on annual sums."""
    sp = SPECIATION[speciation_label]
    s = cells.sum(numeric_only=True)
    res = dict(speciation=speciation_label)
    res["PSO4_over_PNO3_found"] = float(s["PSO4"] / s["PNO3"]) if s.get("PNO3", 0) > 0 else None
    res["PSO4_over_PNO3_expected"] = sp["f_so4"] / sp["f_no3"]
    pm25_g = s["PSO4"] / sp["f_so4"]
    present = [v for v in sp["pm_vars"] if v in s]
    res["species_sum_over_PM25"] = float(s[present].sum() / pm25_g) if pm25_g > 0 else None
    res["ok_ratio"] = (res["PSO4_over_PNO3_found"] is not None and
                       abs(res["PSO4_over_PNO3_found"] / res["PSO4_over_PNO3_expected"] - 1) < 1e-3)
    return res


# ---------------------------------------------------------------------------
# 5. Region: SC municipalities on the BRAIN grid
# ---------------------------------------------------------------------------
def load_municipalities(shp, code_field="CD_MUN", name_field="NM_MUN"):
    import geopandas as gpd
    g = gpd.read_file(shp, encoding="cp1252")          # as FINN_SC python/analysis/_setup.py
    g = g[[code_field, name_field, "geometry"]].rename(columns={code_field: "CD_MUN", name_field: "NM_MUN"})
    g["CD_MUN"] = g["CD_MUN"].astype(str).str.strip()
    return g


def to_index_space(geoms, grid):
    """Lon/lat geometries -> continuous grid coordinates (cell = unit square)."""
    import shapely

    def f(xy):
        x, y = grid.ll2index(xy[:, 0], xy[:, 1])
        return np.column_stack([x, y])
    return shapely.make_valid(shapely.transform(np.asarray(geoms), f))


def region_crop(geoms_idx, grid, margin=3):
    import shapely
    b = shapely.total_bounds(geoms_idx)
    i0 = max(0, int(np.floor(b[0])) - margin)
    j0 = max(0, int(np.floor(b[1])) - margin)
    i1 = min(grid.NCOLS, int(np.ceil(b[2])) + margin)
    j1 = min(grid.NROWS, int(np.ceil(b[3])) + margin)
    return (j0, j1, i0, i1)


def cell_weights(mun, geoms_idx):
    """Fraction of each cell inside each municipality (planar, in grid coordinates).

    Inside one 4 km cell the Mercator scale factor is constant to better than 0.1 %, so the
    planar fraction equals the fraction of the cell's true area.
    """
    import shapely
    rows = []
    for code, name, g in zip(mun["CD_MUN"], mun["NM_MUN"], geoms_idx):
        if g is None or g.is_empty:
            continue
        shapely.prepare(g)
        minx, miny, maxx, maxy = g.bounds
        ii = np.arange(int(np.floor(minx)), int(np.ceil(maxx)))
        jj = np.arange(int(np.floor(miny)), int(np.ceil(maxy)))
        I, J = np.meshgrid(ii, jj)
        I, J = I.ravel(), J.ravel()
        boxes = shapely.box(I, J, I + 1, J + 1)
        a = shapely.area(shapely.intersection(boxes, g))
        k = a > 1e-12
        rows.append(pd.DataFrame({"i": I[k], "j": J[k], "CD_MUN": code, "NM_MUN": name, "frac": a[k]}))
    w = pd.concat(rows, ignore_index=True)
    fsc = w.groupby(["i", "j"], as_index=False)["frac"].sum().rename(columns={"frac": "fSC_raw"})
    fsc["fSC"] = fsc["fSC_raw"].clip(upper=1.0)
    return w, fsc


# ---------------------------------------------------------------------------
# 6. FINN_SC records on the BRAIN grid
# ---------------------------------------------------------------------------
def load_records(path, year):
    df = pd.read_csv(path, low_memory=False)
    df = df[df["year"] == year].copy()
    df["product"] = "FINN " + df["version"].astype(str) + " " + df["sensor"].astype(str)
    df["date"] = pd.Timestamp(f"{year}-01-01") + pd.to_timedelta(df["DAY"].astype(int) - 1, unit="D")
    df["CD_MUN"] = df["CD_MUN"].apply(lambda x: str(int(float(x))) if pd.notna(x) else None)
    return df


def records_to_cells(df, grid, value_cols, keys=("product",)):
    i, j, inside = grid.ll2ij(df["LONGI"].values, df["LATI"].values)
    d = df.assign(i=i, j=j, inside=inside)
    n_out = int((~inside).sum())
    cells = d[inside].groupby(list(keys) + ["date", "i", "j"], as_index=False)[list(value_cols)].sum()
    return cells, n_out


# ---------------------------------------------------------------------------
# 7. FINN v1.5 text files (processing check)
# ---------------------------------------------------------------------------
def _open_text(path):
    """Text stream of a FINN file: tar.gz (largest member) or plain .gz (latin1, as finn2cmaq)."""
    if tarfile.is_tarfile(path):
        tf = tarfile.open(path, "r:*")
        members = [m for m in tf.getmembers() if m.isfile() and not m.name.endswith(".pdf")]
        m = max(members, key=lambda x: x.size)
        return io.TextIOWrapper(tf.extractfile(m), encoding="latin1"), tf
    return io.TextIOWrapper(gzip.open(path, "rb"), encoding="latin1"), None


def _fix_d_exponent(txt):
    return txt.replace("D+", "e+").replace("D-", "e-").replace("d+", "e+").replace("d-", "e-")


def read_finn15(path, lon_rng, lat_rng, names=None, lines_per_chunk=400_000):
    """Rows of a FINN v1.5 file inside a lon/lat box.

    names=None: annual file, columns FINN_V15_COLNAMES (first line is a header and is skipped).
    names='header': NRT daily file, column names taken from its own header (finn2cmaq openfinn).
    """
    stream, owner = _open_text(path)
    try:
        header = stream.readline()
        if names == "header":
            alias = {"LONGITUDE": "LONGI", "LATITUDE": "LATI"}
            cols = [alias.get(c.strip().upper(), c.strip().upper()) for c in header.strip().split(",")]
        else:
            cols = FINN_V15_COLNAMES
        parts = []
        while True:
            lines = list(itertools.islice(stream, lines_per_chunk))
            if not lines:
                break
            txt = _fix_d_exponent("".join(lines))
            ch = pd.read_csv(io.StringIO(txt), header=None, names=cols, index_col=False,
                             skipinitialspace=True)
            lon, lat = ch["LONGI"], ch["LATI"]
            parts.append(ch[(lon >= lon_rng[0]) & (lon <= lon_rng[1]) &
                            (lat >= lat_rng[0]) & (lat <= lat_rng[1])])
    finally:
        stream.close()
        if owner is not None:
            owner.close()
    out = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=cols)
    return out, header.strip()


def finn15_to_tonnes(df, year, molar_mass):
    """FINN v1.5 GEOS-Chem rows (gases mole/day; OC, BC, PM25 kg/day) -> tonnes, BRAIN species."""
    mm = dict(DEFAULT_MOLAR_MASS)
    mm.update(molar_mass or {})
    out = pd.DataFrame({
        "date": pd.Timestamp(f"{year}-01-01") + pd.to_timedelta(df["DAY"].astype(int) - 1, unit="D"),
        "LONGI": df["LONGI"].astype(float), "LATI": df["LATI"].astype(float),
        "AREA": df["AREA"].astype(float)})
    out["CO"] = df["CO"] * mm["CO"] / 1e6
    out["NOx"] = (df["NO"] + df["NO2"]) * mm["NO"] / 1e6
    out["SO2"] = df["SO2"] * mm["SO2"] / 1e6
    out["NH3"] = df["NH3"] * mm["NH3"] / 1e6
    out["CH4"] = df["CH4"] * mm["CH4"] / 1e6
    out["PM25"] = df["PM25"] / 1e3
    out["OC"] = df["OC"] / 1e3
    out["BC"] = df["BC"] / 1e3
    return out


# ---------------------------------------------------------------------------
# 8. Synthetic BRAIN file (numpy replica of finn2cmaq) for the self-test
# ---------------------------------------------------------------------------
def write_synthetic_brain_file(rows, grid, date, out_path, profile_id=210,
                               layerfrac=DEFAULT_LAYERFRAC,
                               speciation_label="finn2cmaq <= 2022-08-04 (BRAIN Table S3)"):
    """Build one CMAQ-ready daily file the way txt2daily.py + daily2hourly3d.py do.

    rows: one FINN day with LONGI, LATI, AREA, CO, NO, NO2, SO2, NH3, CH4 (mole/day) and
    OC, BC, PM25 (kg/day). Returns the daily gridded input (dict of 2D arrays).
    """
    import netCDF4
    sp = SPECIATION[speciation_label]
    i, j, inside = grid.ll2ij(rows["LONGI"].values, rows["LATI"].values)
    keep = inside & (rows["AREA"].values >= FINN2CMAQ_AREA_MIN)
    shape = (grid.NROWS, grid.NCOLS)
    D = {}
    for k in ["CO", "NO", "NO2", "SO2", "NH3", "CH4", "OC", "BC", "PM25"]:
        a = np.zeros(shape)
        np.add.at(a, (j[keep], i[keep]), rows[k].values[keep].astype("f8"))
        D[k] = a
    spc = {"CO": D["CO"], "NO": D["NO"], "NO2": D["NO2"], "SO2": D["SO2"], "NH3": D["NH3"],
           "ECH4": D["CH4"], "POC": D["OC"] * 1000, "PEC": D["BC"] * 1000,
           "PSO4": D["PM25"] * sp["f_so4"] * 1000, "PNO3": D["PM25"] * sp["f_no3"] * 1000}
    spc["PMOTHR"] = D["PM25"] * 1000 - spc["POC"] - spc["PEC"] - spc["PSO4"] - spc["PNO3"]
    I, J = np.meshgrid(np.arange(grid.NCOLS), np.arange(grid.NROWS))
    lon, _ = grid.ij2ll(I, J)
    tz = tz_offset(lon)
    tf = np.zeros((25,) + shape)
    for z in np.unique(tz):
        tf[:, tz == z] = hourly_factors(profile_id, z)[:, None]
    lf = np.asarray(layerfrac, "f8")
    nz = lf.size
    date = pd.Timestamp(date)
    d0 = int(date.strftime("%Y%j"))
    d1 = int((date + pd.Timedelta(days=1)).strftime("%Y%j"))
    with netCDF4.Dataset(out_path, "w", format="NETCDF4_CLASSIC") as ds:
        ds.createDimension("TSTEP", None)
        ds.createDimension("DATE-TIME", 2)
        ds.createDimension("LAY", nz)
        ds.createDimension("VAR", len(spc))
        ds.createDimension("ROW", grid.NROWS)
        ds.createDimension("COL", grid.NCOLS)
        for k in GRID_KEYS:
            ds.setncattr(k, getattr(grid, k))
        ds.setncattr("SDATE", np.int32(d0))
        ds.setncattr("STIME", np.int32(0))
        ds.setncattr("TSTEP", np.int32(10000))
        ds.setncattr("NLAYS", np.int32(nz))
        ds.setncattr("NVARS", np.int32(len(spc)))
        ds.setncattr("FILEDESC", "SYNTHETIC test file (numpy replica of finn2cmaq)")
        tfl = ds.createVariable("TFLAG", "i4", ("TSTEP", "VAR", "DATE-TIME"))
        flags = np.array([[d0, h * 10000] for h in range(24)] + [[d1, 0]], dtype="i4")
        tfl[:] = np.repeat(flags[:, None, :], len(spc), axis=1)
        for name, day in spc.items():
            v = ds.createVariable(name, "f4", ("TSTEP", "LAY", "ROW", "COL"), zlib=True, complevel=1)
            v.units = ("moles/s" if name in GAS_VARS else "g/s").ljust(16)
            v.long_name = name.ljust(16)
            for t in range(25):
                v[t] = (day[None, :, :] * tf[t][None, :, :] * lf[:, None, None] / 3600.0).astype("f4")
    return spc


# ---------------------------------------------------------------------------
# 9. Statistics
# ---------------------------------------------------------------------------
def spearman(x, y):
    from scipy.stats import spearmanr
    x, y = np.asarray(x, "f8"), np.asarray(y, "f8")
    if len(x) < 3 or np.all(x == x[0]) or np.all(y == y[0]):
        return np.nan
    return float(spearmanr(x, y)[0])


def pearson(x, y):
    x, y = np.asarray(x, "f8"), np.asarray(y, "f8")
    if len(x) < 3 or x.std() == 0 or y.std() == 0:
        return np.nan
    return float(np.corrcoef(x, y)[0, 1])


def moving_block_bootstrap(x, y, func, block=7, n_boot=2000, seed=20261101, alpha=0.05):
    """Percentile CI of func(x, y) by the moving-block bootstrap (Kuensch, 1989).

    Pairs (x_t, y_t) are resampled in blocks of `block` consecutive days, which keeps the
    serial dependence of daily fire series inside each block.
    """
    x, y = np.asarray(x, "f8"), np.asarray(y, "f8")
    n = len(x)
    if n < 2 * block:
        return (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    nb = int(np.ceil(n / block))
    starts = rng.integers(0, n - block + 1, size=(n_boot, nb))
    idx = (starts[:, :, None] + np.arange(block)[None, None, :]).reshape(n_boot, -1)[:, :n]
    stats = np.array([func(x[k], y[k]) for k in idx])
    stats = stats[np.isfinite(stats)]
    if stats.size < 0.9 * n_boot:
        return (np.nan, np.nan)
    return (float(np.quantile(stats, alpha / 2)), float(np.quantile(stats, 1 - alpha / 2)))


def save_json(obj, path):
    def conv(o):
        if isinstance(o, (np.integer,)):
            return int(o)
        if isinstance(o, (np.floating,)):
            return float(o)
        if isinstance(o, (pd.Timestamp,)):
            return str(o)
        if isinstance(o, np.ndarray):
            return o.tolist()
        return str(o)
    with open(path, "w", encoding="utf-8") as fh:
        json.dump(obj, fh, indent=2, default=conv, ensure_ascii=False)

In [ ]:
import importlib, glob, json, shutil, time, platform
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import brain_core as bc
importlib.reload(bc)
try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = lambda x, **k: x
OUT = {k: os.path.join(DRIVE_DIR, k) for k in ['cache_days', 'tables', 'figures', 'logs']}
for d in OUT.values():
    os.makedirs(d, exist_ok=True)
SPECIES = bc.SPECIES
pd.set_option('display.width', 160, 'display.max_columns', 30)
print('brain_core', bc.__version__)

## 2. Inputs: FINN_SC repository, URL list, molar masses

In [ ]:
def has_repo(p):
    return os.path.exists(os.path.join(p, 'config', 'config.yml'))

if not has_repo(FINN_SC_DIR):
    r = subprocess.run(['git', 'clone', '--depth', '1', FINN_SC_GIT, '/content/FINN_SC'])
    if r.returncode == 0 and has_repo('/content/FINN_SC'):
        FINN_SC_DIR = '/content/FINN_SC'
    elif IN_COLAB:
        from google.colab import files
        print('Upload the FINN_SC zip')
        up_ = files.upload()
        for name in up_:
            shutil.unpack_archive(name, '/content/FINN_SC_zip')
        hits = [os.path.dirname(os.path.dirname(p)) for p in
                glob.glob('/content/FINN_SC_zip/**/config/config.yml', recursive=True)]
        FINN_SC_DIR = hits[0]
assert has_repo(FINN_SC_DIR), f'FINN_SC not found: {FINN_SC_DIR}'

RECORDS = os.path.join(FINN_SC_DIR, 'data', 'derived', 'finn_sc_records_2003_2025.csv.gz')
SHP     = os.path.join(FINN_SC_DIR, 'data', 'inputs', 'SC_Municipios_2024.shp')
CONFIG  = os.path.join(FINN_SC_DIR, 'config', 'config.yml')

if not os.path.exists(URLS_FILE) and IN_COLAB:
    from google.colab import files
    print('Upload BRAIN_output_urls.txt')
    up_ = files.upload()
    shutil.move(list(up_)[0], URLS_FILE)
assert os.path.exists(URLS_FILE), URLS_FILE

import yaml
cfg = yaml.safe_load(open(CONFIG, encoding='utf-8'))
mm_cfg = cfg['conversion']['molar_mass_g_per_mol']
MM = {k: float(mm_cfg[k]) for k in ['CO', 'NO', 'NO2', 'SO2', 'NH3', 'CH4']}
print('molar masses (g/mol) from FINN_SC config:', MM)

INPUTS = {p: bc.md5(p) for p in [RECORDS, SHP, CONFIG, URLS_FILE]}
for p, h in INPUTS.items():
    print(h, p)

## 3a. Self-tests (about 1 minute)

1. The grid function of `brain_core` against **PseudoNetCDF itself** on 200 000 random points (skipped, with a
   message, if PseudoNetCDF cannot be installed).
2. A **synthetic BRAIN file** built with a numpy copy of finn2cmaq from known fires, read back with the same
   function used for the real files: the daily totals must come back as 1.01 × the input (layer fractions),
   PM₂.₅ must be recovered from `PSO4`, and the time profile and the layers must be recognised.

In [ ]:
G0 = bc.Grid.from_attrs(bc.EXPECTED_GRID, R=EARTH_RADIUS_M)
rng = np.random.default_rng(SEED)
lon_t, lat_t = rng.uniform(-54.2, -47.8, 200_000), rng.uniform(-29.8, -25.5, 200_000)
try:
    if IN_COLAB:   # --no-deps: PseudoNetCDF pins numpy<2 and must not downgrade the Colab numpy
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'pseudonetcdf'], check=True)
    import PseudoNetCDF as pnc
    e = bc.EXPECTED_GRID
    with open('/content/GRIDDESC_SC' if IN_COLAB else 'GRIDDESC_SC', 'w') as fh:
        fh.write("' '\n'SC_MERC'\n  %d %r %r %r %r %r\n' '\n'SC_2019'\n'SC_MERC' %r %r %r %r %d %d 1\n' '\n" % (
            e['GDTYP'], e['P_ALP'], e['P_BET'], e['P_GAM'], e['XCENT'], e['YCENT'],
            e['XORIG'], e['YORIG'], e['XCELL'], e['YCELL'], e['NCOLS'], e['NROWS']))
    gf = pnc.pncopen(fh.name, format='griddesc', GDNAM='SC_2019')
    ip, jp = gf.ll2ij(lon_t, lat_t, clean='mask')
    i, j, inside = G0.ll2ij(lon_t, lat_t)
    same = (np.array_equal(~np.ma.getmaskarray(ip) & ~np.ma.getmaskarray(jp), inside)
            and np.array_equal(np.ma.filled(ip, -9)[inside], i[inside])
            and np.array_equal(np.ma.filled(jp, -9)[inside], j[inside]))
    print('PseudoNetCDF', pnc.__version__, '| same cell for all 200 000 points:', same)
    assert same, 'grid replica differs from PseudoNetCDF: stop and check EARTH_RADIUS_M / IOAPI_ISPH'
except AssertionError:
    raise
except Exception as exc:
    print('PseudoNetCDF test skipped (the replica was checked against PseudoNetCDF 3.5.0 and 3.2.2 when this '
          'notebook was written):', repr(exc)[:200])

n = 300
syn = pd.DataFrame({'LONGI': rng.uniform(-53.8, -48.4, n), 'LATI': rng.uniform(-29.3, -26.0, n),
                    'AREA': rng.choice([30.0, 1e6], n, p=[0.05, 0.95])})
for k in ['CO', 'NO', 'NO2', 'SO2', 'NH3', 'CH4']:
    syn[k] = rng.lognormal(10, 1, n)
for k in ['OC', 'BC']:
    syn[k] = rng.lognormal(5, 1, n)
syn['PM25'] = 3 * (syn['OC'] + syn['BC'])
spec0 = 'finn2cmaq <= 2022-08-04 (BRAIN Table S3)'
test_nc = os.path.join(NC_TMP, '_selftest.nc')
bc.write_synthetic_brain_file(syn, G0, f'{YEAR}-07-15', test_nc, profile_id=210)
crop_t = (0, G0.NROWS, 0, G0.NCOLS)
c_t, d_t = bc.extract_day(test_nc, f'{YEAR}-07-15', crop_t, bc.needed_vars(spec0))
keep = (syn.AREA >= bc.FINN2CMAQ_AREA_MIN).values & G0.ll2ij(syn.LONGI, syn.LATI)[2]
ratios = {k: c_t[v].sum() / syn.loc[keep, k].sum() for k, v in [('CO', 'CO'), ('CH4', 'ECH4'), ('NO2', 'NO2')]}
pm = bc.brain_to_tonnes(c_t, MM, spec0)['PM25'].sum() / (syn.loc[keep, 'PM25'].sum() / 1e3)
lon_mx, _ = G0.ij2ll(*d_t['maxcell_ij'])
tp = bc.match_time_profile(d_t['maxcell_hourly25'], bc.tz_offset(lon_mx))
ly = bc.match_layers(d_t['maxcell_layers'])
print('daily total / input:', {k: round(v, 7) for k, v in ratios.items()}, '| PM2.5 recovered / input:', round(pm, 7))
print('time profile found:', tp['best'], tp['matched'], '| layers = default:', ly['matched'])
assert all(abs(v - 1.01) < 1e-5 for v in ratios.values()) and abs(pm - 1.01) < 1e-5
assert tp['best'] == 210 and tp['matched'] and ly['matched'] and d_t['tflag_ok']
os.remove(test_nc)
print('self-tests passed')

## 3b. The URL list

In [ ]:
URLS = bc.parse_brain_urls(URLS_FILE)
all_days = pd.date_range(f'{YEAR}-01-01', f'{YEAR}-12-31', freq='D')
print(f'{len(URLS)} FINN files in the list, {URLS.date.min().date()} to {URLS.date.max().date()}')
no_url = all_days.difference(URLS.date)
print(f'days of {YEAR} without a URL: {len(no_url)}', list(no_url.strftime('%Y-%m-%d'))[:20])
print('other years in the list:', sorted(set(URLS.date.dt.year) - {YEAR}))
URLS = URLS[URLS.date.dt.year == YEAR].reset_index(drop=True)
if ONLY_DATES:
    URLS = URLS[URLS.date.isin(pd.to_datetime(ONLY_DATES))].reset_index(drop=True)
    print('trial run with', len(URLS), 'dates')

import requests
SESSION = requests.Session()
SESSION.headers['User-Agent'] = 'Mozilla/5.0 (FINN_SC BRAIN comparison notebook)'

def get_zip(row):
    # Use a valid local copy if there is one, otherwise download.
    for d in [MANUAL_ZIP_DIR, os.path.join(DRIVE_DIR, 'zips'), ZIP_DIR]:
        p = os.path.join(d, row.fileName)
        if bc.valid_zip(p):
            return p
    p = bc.download(row.url, os.path.join(ZIP_DIR, row.fileName), session=SESSION)
    time.sleep(PAUSE_S)
    if KEEP_ZIPS_ON_DRIVE:
        os.makedirs(os.path.join(DRIVE_DIR, 'zips'), exist_ok=True)
        shutil.copy2(p, os.path.join(DRIVE_DIR, 'zips', row.fileName))
    return p

## 3c. First file: structure, grid, species and units

Prints what the BRAIN file actually contains and **stops** if the grid, the time axis, the species or the units
differ from what the rest of the notebook expects.

In [ ]:
row0 = URLS.iloc[0]
try:
    z0 = get_zip(row0)
except Exception:
    print(f'Could not download {row0.fileName}. Open its URL in a browser; if the browser gets the zip, put the '
          f'zips in {MANUAL_ZIP_DIR} (same names) and run again. URL:\n{row0.url}')
    raise
nc0, members0 = bc.extract_nc(z0, NC_TMP)
H = bc.read_header(nc0)
print('zip members:', members0, '| netCDF data model:', H['data_model'])
print('dimensions:', H['dims'])
for k in ['GDNAM', 'GDTYP', 'P_ALP', 'P_BET', 'P_GAM', 'XCENT', 'YCENT', 'XORIG', 'YORIG', 'XCELL', 'YCELL',
          'NCOLS', 'NROWS', 'NLAYS', 'NVARS', 'SDATE', 'STIME', 'TSTEP', 'VGTYP', 'VGTOP', 'UPNAM']:
    print(f'  {k:6s} = {H["attrs"].get(k)}')
print('  VGLVLS =', H['attrs'].get('VGLVLS'))
print('  FILEDESC:', str(H['attrs'].get('FILEDESC', ''))[:300])
print('  HISTORY :', str(H['attrs'].get('HISTORY', ''))[:600])
VT = pd.DataFrame([dict(var=k, units=v['units'], long_name=v['long_name'], var_desc=v['var_desc'][:60],
                        shape=v['shape']) for k, v in H['variables'].items()])
display(VT) if IN_COLAB else print(VT.to_string())
print('TFLAG first/last:', H['tflag'][0], H['tflag'][-1], '| steps:', len(H['tflag']))

GRID_ATTRS = {k: H['attrs'][k] for k in bc.GRID_KEYS}
mism = bc.compare_grid(H['attrs'])
print('grid differences from the BRAIN MEGAN file of the same domain:', mism or 'none')
assert not mism, 'the FINN file is not on the expected SC_2019 grid'
GRID = bc.Grid.from_attrs(H['attrs'], R=EARTH_RADIUS_M)

SPEC = bc.detect_speciation(H['variables'])
VARS = bc.needed_vars(SPEC)
unit_problems = bc.check_units(H['variables'], VARS)
print('speciation:', SPEC)
print('variables read:', VARS)
print('unit problems:', unit_problems or 'none')
assert not unit_problems, 'units or variables differ from finn2cmaq output: stop and inspect the table above'
print('CO2 present:', 'CO2' in H['variables'], '| PM10/PMC present:', 'PMC' in H['variables'])
bc.save_json(H, os.path.join(OUT['logs'], 'header_first_file.json'))
os.remove(nc0)

## 4. Santa Catarina on the BRAIN grid

Each municipality (IBGE 2024) is projected with the BRAIN grid projection and intersected with the 4 km cells.
`fSC` is the fraction of a cell inside the State; region R1 is the set of cells with `fSC ≥ 0.999`.

In [ ]:
MUN = bc.load_municipalities(SHP)
GEOMS_IDX = bc.to_index_space(MUN.geometry.values, GRID)
CROP = bc.region_crop(GEOMS_IDX, GRID, margin=CROP_MARGIN_CELLS)
W, FSC = bc.cell_weights(MUN, GEOMS_IDX)
FSC['interior'] = FSC['fSC'] >= INTERIOR_MIN_FRACTION
j0, j1, i0, i1 = CROP
assert W.i.between(i0, i1 - 1).all() and W.j.between(j0, j1 - 1).all()
print(f'{len(MUN)} municipalities | crop rows {j0}-{j1 - 1}, cols {i0}-{i1 - 1} '
      f'({(j1 - j0) * (i1 - i0)} cells read per file)')
print(f'cells touching SC: {len(FSC)} | interior (R1): {FSC.interior.sum()} | border: {(~FSC.interior).sum()}')
print(f'sum of municipal fractions per cell: max {FSC.fSC_raw.max():.6f} (above 1 = overlapping polygons)')
print(f'SC area in cell units: {FSC.fSC.sum():.1f}')
W.to_csv(os.path.join(OUT['tables'], 'cell_municipality_fractions.csv'), index=False)
FSC.to_csv(os.path.join(OUT['tables'], 'cell_fraction_SC.csv'), index=False)

import shapely
SC_IDX = shapely.union_all(GEOMS_IDX)

def _rings(geoms):
    for g in geoms:
        for poly in getattr(g, 'geoms', [g]):
            if poly.geom_type == 'Polygon':
                yield poly.exterior.xy

def sc_outline(ax, lw=0.6, municipalities=True):
    if municipalities:
        for x, y in _rings(GEOMS_IDX):
            ax.plot(np.asarray(x) - i0, np.asarray(y) - j0, color='0.6', lw=0.15)
    for x, y in _rings([SC_IDX]):
        ax.plot(np.asarray(x) - i0, np.asarray(y) - j0, color='k', lw=lw)

def to_map(df, col):
    m = np.full((j1 - j0, i1 - i0), np.nan)
    m[df.j.values - j0, df.i.values - i0] = df[col].values
    return m

fig, ax = plt.subplots(figsize=(7, 4.6))
im = ax.imshow(to_map(FSC, 'fSC'), origin='lower', cmap='Greens', vmin=0, vmax=1)
sc_outline(ax)
ax.set_title('Fraction of each BRAIN 4 km cell inside SC')
ax.set_xlabel('column - %d' % i0); ax.set_ylabel('row - %d' % j0)
fig.colorbar(im, ax=ax, shrink=0.8, label='fSC')
fig.savefig(os.path.join(OUT['figures'], 'cell_fraction_SC.png'), dpi=200, bbox_inches='tight')
plt.show()

## 5. Extract every BRAIN day

For each date: download (or reuse) the zip, unzip, read the SC window, add the 24 hourly rates of the file's own
date × 3600 s over all layers, and save the non-zero cells to `cache_days/<date>.csv.gz` with a diagnostics file.
Days already in the cache are skipped. Failures are listed in `logs/failures.csv` and the loop goes on.

In [ ]:
done = {os.path.basename(p)[:10] for p in glob.glob(os.path.join(OUT['cache_days'], '*.json'))}
todo = URLS[~URLS.date.dt.strftime('%Y-%m-%d').isin(done)]
print(f'{len(done)} days in the cache, {len(todo)} to process')
failures = []
for row in tqdm(list(todo.itertuples()), desc='BRAIN days'):
    key = row.date.strftime('%Y-%m-%d')
    nc = None
    try:
        z = get_zip(row)
        nc, members = bc.extract_nc(z, NC_TMP)
        cells, diag = bc.extract_day(nc, row.date, CROP, VARS, expected_grid=GRID_ATTRS)
        diag.update(zip=row.fileName, zip_bytes=os.path.getsize(z), zip_md5=bc.md5(z), zip_members=members)
        cells.to_csv(os.path.join(OUT['cache_days'], f'{key}.csv.gz'), index=False)
        bc.save_json(diag, os.path.join(OUT['cache_days'], f'{key}.json'))
    except Exception as exc:  # keep going, report at the end
        failures.append(dict(date=key, url=row.url, error=repr(exc)[:400]))
        print('FAILED', key, repr(exc)[:200])
    finally:
        if nc and os.path.exists(nc):
            os.remove(nc)
pd.DataFrame(failures, columns=['date', 'url', 'error']).to_csv(os.path.join(OUT['logs'], 'failures.csv'), index=False)
print('failures:', len(failures))

### 5b. Diagnostics of all processed days

Checks the time axis, the grid, missing or negative values, the hourly profile and the layer fractions, and
derives the factor between a BRAIN daily total and its FINN input.

In [ ]:
DIAG = [json.load(open(p, encoding='utf-8')) for p in sorted(glob.glob(os.path.join(OUT['cache_days'], '*.json')))]
DAYS_OK = pd.DatetimeIndex(sorted(pd.Timestamp(d['date']) for d in DIAG))
print(f'processed days: {len(DAYS_OK)} of {len(all_days)}')
missing_days = all_days.difference(DAYS_OK)
print('days without BRAIN data (excluded from EVERY dataset below):', len(missing_days),
      list(missing_days.strftime('%Y-%m-%d'))[:30])
dg = pd.DataFrame(DIAG)
print('TFLAG ok on every day:', bool(dg.tflag_ok.all()), '| next-day 00Z step present:', bool(dg.has_next_00z.all()))
print('grid mismatches:', int(dg.grid_mismatch.map(len).sum()), '| missing variables:',
      sorted({v for x in dg.missing_vars for v in x}) or 'none')
print('non-finite values:', {k: v for x in dg.nonfinite for k, v in x.items()} or 'none')
neg = pd.DataFrame([dict(date=d['date'], var=k, n=v[0], amount=v[1]) for d in DIAG for k, v in d['negatives'].items()])
print('negative values:', 'none' if neg.empty else neg.groupby('var').agg(days=('date', 'nunique'), cells=('n', 'sum'), amount=('amount', 'sum')))
print('layers per file:', sorted(dg.nlay.dropna().unique()))
r25 = (dg.CO_all25_mol / dg.CO_day_mol).replace([np.inf, -np.inf], np.nan).dropna()
print(f'25-step / 24-step CO: median {r25.median():.6f} (profile 210 gives 1.005331)')

prof, lay = [], []
for d in DIAG:
    if 'maxcell_ij' not in d:
        continue
    lon_c, _ = GRID.ij2ll(*d['maxcell_ij'])
    m = bc.match_time_profile(d['maxcell_hourly25'], bc.tz_offset(lon_c))
    prof.append(dict(date=d['date'], best=m['best'], matched=m['matched'], S24=m.get('S24'),
                     diff=min(m['max_abs_diff'].values())))
    lay.append(bc.match_layers(d['maxcell_layers']))
prof, lay = pd.DataFrame(prof), pd.DataFrame(lay)
print('hourly profile recognised:', prof.groupby(['best', 'matched']).size().to_dict())
print('layer fractions = finn2cmaq default:', lay.matched.value_counts().to_dict(), '| default sum:', lay.default_sum.unique())
PROFILE = int(prof.best.mode()[0]) if len(prof) else None
LAYERS_DEFAULT = bool(len(lay) and lay.matched.all())
S24 = float(prof.S24.median()) if len(prof) else np.nan
FACTOR = S24 * (float(lay.default_sum.iloc[0]) if LAYERS_DEFAULT else np.nan)
LAYER_PROFILE = None
if len(lay):
    v_ = np.array(next(d for d in DIAG if 'maxcell_layers' in d)['maxcell_layers'])
    LAYER_PROFILE = (v_ / v_[0]).round(4).tolist()
    print('layer profile found (relative to the first layer):', LAYER_PROFILE)
if LAYERS_DEFAULT:
    print(f'=> BRAIN daily total = FINN input x {FACTOR:.4f} (24-h profile share {S24:.6f} x layer sum)')
else:
    print('=> the layer fractions differ in shape from the finn2cmaq default, so their sum cannot be read from the '
          'files alone; section 9a measures the BRAIN / FINN-input factor directly')
prof.to_csv(os.path.join(OUT['logs'], 'time_profile_check.csv'), index=False)

### 5c. BRAIN in tonnes, and the PM₂.₅ identities

In [ ]:
BR_RAW = pd.concat([pd.read_csv(p, parse_dates=['date'])
                    for p in sorted(glob.glob(os.path.join(OUT['cache_days'], '*.csv.gz')))], ignore_index=True)
BR_RAW = BR_RAW[BR_RAW.date.isin(DAYS_OK)]
SPC_CHECK = bc.speciation_checks(BR_RAW, SPEC)
print(SPC_CHECK)
assert SPC_CHECK['ok_ratio'], 'PSO4/PNO3 does not match the detected finn2cmaq speciation'
BR = bc.brain_to_tonnes(BR_RAW, MM, SPEC)
print(f'{len(BR)} cell-days with emissions in the SC window')
print('PM2.5 from PSO4 vs sum of PM species (t):', round(BR.PM25.sum(), 1), round(BR.PM25_species_sum.sum(), 1))

## 6. FINN_SC records of 2019 on the BRAIN grid

In [ ]:
REC = bc.load_records(RECORDS, YEAR)
for s in SPECIES:
    REC[s] = REC[bc.OURS_COLUMN[s]].astype(float)
PRODUCTS = sorted(REC['product'].unique())
print(REC.groupby('product').size().rename('records'))
REC = REC[REC.date.isin(DAYS_OK)].copy()
print('records on days with BRAIN data:', REC.groupby('product').size().to_dict())
OURS, n_out = bc.records_to_cells(REC, GRID, SPECIES)
print('records outside the BRAIN domain:', n_out)
assert n_out == 0
assert OURS.i.between(i0, i1 - 1).all() and OURS.j.between(j0, j1 - 1).all()

LABEL = {'BRAIN': 'BRAIN (FINN v1.5 MODIS)'}
LABEL.update({p: p for p in PRODUCTS})
COLOR = {'BRAIN': '#444444', 'FINN v2.5 MODIS': '#d95f02', 'FINN v2.5 MODIS+VIIRS': '#1b6fb3',
         'FINN v1.5 MODIS (annual file)': '#7a7a00'}

## 7. Region totals

`R1` = cells entirely inside SC (identical region). `R2` = the State: BRAIN cells × `fSC`, FINN_SC records in SC.
`R2 low` and `R2 high` bound BRAIN with interior cells only and with every cell touching SC.

In [ ]:
F = FSC.set_index(['i', 'j'])

def with_f(cells):
    c = cells.join(F[['fSC', 'interior']], on=['i', 'j'])
    c['fSC'] = c['fSC'].fillna(0.0)
    c['interior'] = c['interior'].fillna(False).astype(bool)
    return c

def daily(cells, how, cols=SPECIES):
    if how == 'R1':
        c = cells[cells.interior]; v = c[cols]
    elif how == 'R2':
        c = cells[cells.fSC > 0]; v = c[cols].mul(c.fSC, axis=0)
    elif how == 'R2_high':
        c = cells[cells.fSC > 0]; v = c[cols]
    else:
        raise ValueError(how)
    out = v.groupby(c['date']).sum()
    return out.reindex(DAYS_OK, fill_value=0.0)

BRf = with_f(BR)
OURSf = with_f(OURS)
SERIES = {('BRAIN', 'R1'): daily(BRf, 'R1', SPECIES + ['PM25_species_sum']),
          ('BRAIN', 'R2'): daily(BRf, 'R2', SPECIES + ['PM25_species_sum']),
          ('BRAIN', 'R2_low'): daily(BRf, 'R1', SPECIES + ['PM25_species_sum']),
          ('BRAIN', 'R2_high'): daily(BRf, 'R2_high', SPECIES + ['PM25_species_sum'])}
for p in PRODUCTS:
    SERIES[(p, 'R1')] = daily(OURSf[OURSf['product'] == p], 'R1')
    SERIES[(p, 'R2')] = REC[REC['product'] == p].groupby('date')[SPECIES].sum().reindex(DAYS_OK, fill_value=0.0)

ann = pd.DataFrame({f'{d} | {r}': s[SPECIES].sum() for (d, r), s in SERIES.items()}).T
ann.index.name = 'dataset | region'
rat = []
for p in PRODUCTS:
    for r in ['R1', 'R2']:
        rat.append((ann.loc[f'{p} | {r}'] / ann.loc[f'BRAIN | {r}']).rename(f'{p} / BRAIN | {r}'))
    rat.append((ann.loc[f'{p} | R2'] / ann.loc['BRAIN | R2_high']).rename(f'{p} / BRAIN | R2 (BRAIN high)'))
    rat.append((ann.loc[f'{p} | R2'] / ann.loc['BRAIN | R2_low']).rename(f'{p} / BRAIN | R2 (BRAIN low)'))
ANNUAL = pd.concat([ann, pd.DataFrame(rat)])
ANNUAL.to_csv(os.path.join(OUT['tables'], 'annual_totals_and_ratios.csv'))
print(f'Annual totals (t) over {len(DAYS_OK)} days, and ratios')
print(ANNUAL.round(3).to_string())
print('\nBRAIN PM2.5 as the sum of all PM species (t): R2 =', round(SERIES[('BRAIN', 'R2')]['PM25_species_sum'].sum(), 1))
HEAD = pd.DataFrame({p: {f'{s} {r}': ANNUAL.loc[f'{p} / BRAIN | {r}', s] for s in ['CO', 'PM25'] for r in ['R1', 'R2']}
                     for p in PRODUCTS}).T
HEAD['CO R2 range'] = [f"{ANNUAL.loc[f'{p} / BRAIN | R2 (BRAIN high)', 'CO']:.2f}-{ANNUAL.loc[f'{p} / BRAIN | R2 (BRAIN low)', 'CO']:.2f}"
                       for p in PRODUCTS]
print('\nHeadline: dataset / BRAIN'); print(HEAD.round(3).to_string())

## 8. Months, days, cells and municipalities

In [ ]:
# ---- 8a. Months ------------------------------------------------------------
mon = {k: s[SPECIES].groupby(s.index.month).sum() for k, s in SERIES.items() if k[1] in ('R1', 'R2')}
MONTHLY = pd.concat({f'{d} | {r}': m for (d, r), m in mon.items()}, names=['dataset | region', 'month'])
MONTHLY.to_csv(os.path.join(OUT['tables'], 'monthly_totals.csv'))
rows = []
for p in PRODUCTS:
    for r in ['R1', 'R2']:
        a, b = mon[('BRAIN', r)], mon[(p, r)]
        for s in SPECIES:
            rows.append(dict(product=p, region=r, species=s, n_months=len(a), pearson=bc.pearson(a[s], b[s]),
                             spearman=bc.spearman(a[s], b[s]),
                             peak_month_BRAIN=int(a[s].idxmax()), peak_month_product=int(b[s].idxmax())))
MONTH_STATS = pd.DataFrame(rows)
MONTH_STATS.to_csv(os.path.join(OUT['tables'], 'monthly_correlations.csv'), index=False)
print(MONTH_STATS[MONTH_STATS.region == 'R2'].round(3).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, s in zip(axes, ['CO', 'PM25']):
    keys = ['BRAIN'] + PRODUCTS
    w = 0.8 / len(keys)
    for n_, k in enumerate(keys):
        y = mon[(k, 'R2')][s] / 1e3
        ax.bar(y.index + (n_ - (len(keys) - 1) / 2) * w, y.values, width=w, color=COLOR.get(k), label=LABEL[k])
    ax.set_title(f'{bc.SPECIES_LABEL[s]}, Santa Catarina (R2), {YEAR}')
    ax.set_xticks(range(1, 13)); ax.set_xlabel('month'); ax.set_ylabel('kt')
    ax.spines[['top', 'right']].set_visible(False)
axes[0].legend(frameon=False, fontsize=8)
fig.tight_layout(); fig.savefig(os.path.join(OUT['figures'], 'monthly_CO_PM25.png'), dpi=200); plt.show()

In [ ]:
# ---- 8b. Days --------------------------------------------------------------
rows = []
for p in PRODUCTS:
    for r in ['R1', 'R2']:
        a, b = SERIES[('BRAIN', r)], SERIES[(p, r)]
        for s in SPECIES:
            x, y = a[s].values, b[s].values
            lo, hi = bc.moving_block_bootstrap(x, y, bc.spearman, block=BLOCK_DAYS, n_boot=N_BOOT, seed=SEED)
            fire = (x > 0) | (y > 0)
            both, only_b, only_p = int(((x > 0) & (y > 0)).sum()), int(((x > 0) & (y == 0)).sum()), int(((x == 0) & (y > 0)).sum())
            rows.append(dict(product=p, region=r, species=s, n_days=len(x), spearman=bc.spearman(x, y),
                             ci95_low=lo, ci95_high=hi, spearman_fire_days=bc.spearman(x[fire], y[fire]),
                             days_both=both, days_only_BRAIN=only_b, days_only_product=only_p,
                             days_none=int((~fire).sum()), jaccard=both / max(1, both + only_b + only_p)))
DAY_STATS = pd.DataFrame(rows)
DAY_STATS.to_csv(os.path.join(OUT['tables'], 'daily_statistics.csv'), index=False)
pd.concat({f'{d} | {r}': s for (d, r), s in SERIES.items()}, axis=1).to_csv(os.path.join(OUT['tables'], 'daily_totals.csv'))
print(DAY_STATS[DAY_STATS.species.isin(['CO', 'PM25'])].round(3).to_string(index=False))

fig, ax = plt.subplots(figsize=(11, 3.4))
for k in ['BRAIN'] + PRODUCTS:
    y = SERIES[(k, 'R2')]['CO']
    ax.plot(y.index, y.values, lw=0.9, color=COLOR.get(k), label=LABEL[k])
ax.set_yscale('symlog', linthresh=10); ax.set_ylabel('CO (t/day)')
ax.set_title(f'Daily CO, Santa Catarina (R2), {YEAR}')
ax.spines[['top', 'right']].set_visible(False); ax.legend(frameon=False, fontsize=8)
fig.tight_layout(); fig.savefig(os.path.join(OUT['figures'], 'daily_CO.png'), dpi=200); plt.show()

In [ ]:
# ---- 8c. Cells (region R1: identical cells for every dataset) -------------
def annual_cells(cells):
    return cells[cells.interior].groupby(['i', 'j'])[SPECIES].sum()

AC = {'BRAIN': annual_cells(BRf)}
for p in PRODUCTS:
    AC[p] = annual_cells(OURSf[OURSf['product'] == p])
rows = []
for p in PRODUCTS:
    u = AC['BRAIN'].join(AC[p], how='outer', lsuffix='_B', rsuffix='_P').fillna(0.0)
    for s in SPECIES:
        x, y = u[f'{s}_B'], u[f'{s}_P']
        rows.append(dict(product=p, species=s, n_cells_either=len(u), n_cells_BRAIN=int((x > 0).sum()),
                         n_cells_product=int((y > 0).sum()), n_cells_both=int(((x > 0) & (y > 0)).sum()),
                         spearman_cells=bc.spearman(x, y),
                         share_BRAIN_in_cells_with_product=float(x[y > 0].sum() / x.sum()) if x.sum() else np.nan,
                         share_product_in_cells_with_BRAIN=float(y[x > 0].sum() / y.sum()) if y.sum() else np.nan))
CELL_STATS = pd.DataFrame(rows)
CELL_STATS.to_csv(os.path.join(OUT['tables'], 'cell_statistics_R1.csv'), index=False)
pd.concat(AC, axis=1).to_csv(os.path.join(OUT['tables'], 'cell_annual_R1.csv'))
print(CELL_STATS.round(3).to_string(index=False))

# maps of annual CO per cell (all cells touching SC; BRAIN border cells include fires outside SC)
maps = {'BRAIN': BRf[BRf.fSC > 0].groupby(['i', 'j'])['CO'].sum().reset_index()}
for p in PRODUCTS:
    maps[p] = OURSf[OURSf['product'] == p].groupby(['i', 'j'])['CO'].sum().reset_index()
vmax = max(np.log10(m.CO.max()) for m in maps.values() if len(m))
fig, axes = plt.subplots(1, len(maps), figsize=(5.6 * len(maps), 4.4))
for ax, (k, m) in zip(np.atleast_1d(axes), maps.items()):
    mm_ = to_map(m.assign(logCO=np.log10(m.CO.where(m.CO > 0))), 'logCO')
    im = ax.imshow(mm_, origin='lower', cmap='magma_r', vmin=-1, vmax=vmax)
    sc_outline(ax)
    ax.set_title(LABEL[k], fontsize=9); ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(im, ax=axes, shrink=0.8, label=f'log10 CO (t per 4 km cell, {YEAR})')
fig.savefig(os.path.join(OUT['figures'], 'map_CO_cells.png'), dpi=200, bbox_inches='tight'); plt.show()

In [ ]:
# ---- 8d. Municipalities (BRAIN by cell-area fractions; FINN_SC by point location) ----
br_cell = BR.groupby(['i', 'j'])[SPECIES].sum().reset_index()
br_mun = (W.merge(br_cell, on=['i', 'j'])
           .assign(**{s: lambda d, s=s: d[s] * d['frac'] for s in SPECIES})
           .groupby('CD_MUN')[SPECIES].sum())
MUNI = MUN[['CD_MUN', 'NM_MUN']].set_index('CD_MUN').join(br_mun.add_prefix('BRAIN_'))
for p in PRODUCTS:
    MUNI = MUNI.join(REC[REC['product'] == p].groupby('CD_MUN')[SPECIES].sum().add_prefix(f'{p}_'))
MUNI = MUNI.fillna(0.0)
MUNI.to_csv(os.path.join(OUT['tables'], 'municipal_annual.csv'))
print('BRAIN: municipal sum / R2 total (CO) =', round(MUNI['BRAIN_CO'].sum() / SERIES[('BRAIN', 'R2')]['CO'].sum(), 6))
rows = []
for p in PRODUCTS:
    for s in SPECIES:
        x, y = MUNI[f'BRAIN_{s}'], MUNI[f'{p}_{s}']
        rows.append(dict(product=p, species=s, n_municipalities=len(MUNI), spearman=bc.spearman(x, y),
                         top10_overlap=len(set(x.nlargest(10).index) & set(y.nlargest(10).index))))
MUN_STATS = pd.DataFrame(rows)
MUN_STATS.to_csv(os.path.join(OUT['tables'], 'municipal_statistics.csv'), index=False)
print(MUN_STATS.round(3).to_string(index=False))

fig, axes = plt.subplots(1, len(PRODUCTS), figsize=(5 * len(PRODUCTS), 4.2))
for ax, p in zip(np.atleast_1d(axes), PRODUCTS):
    x, y = MUNI['BRAIN_CO'], MUNI[f'{p}_CO']
    ax.scatter(x + 0.1, y + 0.1, s=10, alpha=0.6, color=COLOR.get(p))
    lim = [0.1, max(x.max(), y.max()) * 2]
    ax.plot(lim, lim, color='0.5', lw=0.8); ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlabel('BRAIN CO (t + 0.1)'); ax.set_ylabel(f'{p} CO (t + 0.1)')
    ax.set_title(f'Municipalities of SC, {YEAR}', fontsize=9); ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout(); fig.savefig(os.path.join(OUT['figures'], 'municipal_CO_scatter.png'), dpi=200); plt.show()

## 9. Optional: processing check against the FINN v1.5 input

**9a** reads the FINN v1.5 2019 annual file from NSF NCAR, keeps the fires in the SC window, drops `AREA` < 50 m²
and grids them as finn2cmaq does. Where BRAIN and this file hold the same fires, their ratio must equal the
factor found in 5b (1.01 with the default settings). Fires present in only one of them measure the difference
between the near-real-time input of BRAIN and the final annual file.

**9b** tries the near-real-time daily files themselves (`GLOB_GEOSchem_YYYYJJJ.txt.gz`, the input of
`run_nrt.sh`) for the days with most BRAIN emission. If they are still online, the ratio per cell must equal the
factor everywhere, which verifies the extraction end to end. If they are not online, the cell says so.

In [ ]:
gl_lon, gl_lat = GRID.ij2ll(np.array([i0 - 1, i1, i0 - 1, i1]), np.array([j0 - 1, j0 - 1, j1, j1]))
LON_RNG, LAT_RNG = (min(gl_lon) - 0.1, max(gl_lon) + 0.1), (min(gl_lat) - 0.1, max(gl_lat) + 0.1)

def finn_cells(df_t, label):
    k = df_t['AREA'] >= bc.FINN2CMAQ_AREA_MIN
    print(f'  {label}: {len(df_t)} rows in the window, {int((~k).sum())} dropped for AREA < 50 m2')
    c, _ = bc.records_to_cells(df_t[k].assign(product=label), GRID, SPECIES)
    c = c[c.i.between(i0, i1 - 1) & c.j.between(j0, j1 - 1) & c.date.isin(DAYS_OK)]
    return with_f(c)

def cell_day_match(cf, label, dates=None):
    b = BRf if dates is None else BRf[BRf.date.isin(dates)]
    m = b.merge(cf, on=['date', 'i', 'j'], how='outer', suffixes=('_B', '_F')).fillna(0.0)
    both = (m.CO_B > 0) & (m.CO_F > 0)
    r = (m.CO_B[both] / m.CO_F[both])
    f = FACTOR if np.isfinite(FACTOR) else (float(r.median()) if len(r) else np.nan)
    return dict(dataset=label, days=int(m.date.nunique()), cell_days_BRAIN=int((m.CO_B > 0).sum()), cell_days_file=int((m.CO_F > 0).sum()),
                cell_days_both=int(both.sum()),
                share_BRAIN_CO_in_common_cell_days=float(m.CO_B[both].sum() / m.CO_B.sum()) if m.CO_B.sum() else np.nan,
                share_file_CO_in_common_cell_days=float(m.CO_F[both].sum() / m.CO_F.sum()) if m.CO_F.sum() else np.nan,
                ratio_median=float(r.median()) if len(r) else np.nan,
                ratio_p05=float(r.quantile(0.05)) if len(r) else np.nan,
                ratio_p95=float(r.quantile(0.95)) if len(r) else np.nan,
                share_within_1pct_of_factor=float((np.abs(r / f - 1) < 0.01).mean()) if len(r) else np.nan,
                factor=f, factor_source='5b (default layers)' if np.isfinite(FACTOR) else 'median ratio of this check')

CHECKS = []
if RUN_V15_CHECK:
    try:
        path15 = None
        for u in bc.FINN15_ANNUAL_URLS:
            url = u.format(year=YEAR)
            try:
                path15 = bc.download(url, os.path.join(FINN15_DIR, 'annual', os.path.basename(url)),
                                     session=SESSION, check=bc.is_gzip, retries=2)
                break
            except Exception as exc:
                print('not available:', url, repr(exc)[:150])
        if path15:
            F15, head15 = bc.read_finn15(path15, LON_RNG, LAT_RNG)
            print('header of the v1.5 file:', head15[:200])
            C15 = finn_cells(bc.finn15_to_tonnes(F15, YEAR, MM), 'FINN v1.5 MODIS (annual file)')
            SERIES[('FINN v1.5 MODIS (annual file)', 'R1')] = daily(C15, 'R1')
            SERIES[('FINN v1.5 MODIS (annual file)', 'R2')] = daily(C15, 'R2')
            t15 = pd.DataFrame({r: SERIES[('FINN v1.5 MODIS (annual file)', r)][SPECIES].sum() /
                                    SERIES[('BRAIN', r)][SPECIES].sum() for r in ['R1', 'R2']}).T
            print('FINN v1.5 annual file / BRAIN', f'(expected 1 / factor = {1 / FACTOR:.4f})' if np.isfinite(FACTOR) else '')
            print(t15.round(4).to_string())
            t15.to_csv(os.path.join(OUT['tables'], 'check_v15_annual_over_BRAIN.csv'))
            CHECKS.append(cell_day_match(C15, 'FINN v1.5 annual file'))
            if not np.isfinite(FACTOR):
                FACTOR = CHECKS[-1]['factor']
                print(f'=> measured BRAIN / FINN v1.5 factor: {FACTOR:.4f} (median over {CHECKS[-1]["cell_days_both"]} '
                      f'cell-days; 5th-95th percentile {CHECKS[-1]["ratio_p05"]:.4f}-{CHECKS[-1]["ratio_p95"]:.4f})')
    except Exception as exc:
        print('9a skipped:', repr(exc)[:300])

if RUN_NRT_CHECK:
    top = SERIES[('BRAIN', 'R2')]['CO'].nlargest(NRT_N_DAYS).index
    got, nrt_dates = [], []
    for d in top:
        url = bc.FINN15_NRT_URL.format(yj=d.strftime('%Y%j'))
        try:
            p = bc.download(url, os.path.join(FINN15_DIR, 'nrt', os.path.basename(url)),
                            session=SESSION, check=bc.is_gzip, retries=2)
            df_, head_ = bc.read_finn15(p, LON_RNG, LAT_RNG, names='header')
            got.append(bc.finn15_to_tonnes(df_, YEAR, MM))
            nrt_dates.append(d)
            print('NRT file read:', os.path.basename(url), '| header:', head_[:120])
        except Exception as exc:
            print('NRT file not used:', url, repr(exc)[:150])
    if got:
        CN = finn_cells(pd.concat(got, ignore_index=True), 'FINN v1.5 NRT')
        sub = BRf[BRf.date.isin(nrt_dates)]
        m = sub.merge(CN, on=['date', 'i', 'j'], how='outer', suffixes=('_B', '_F')).fillna(0.0)
        for s in SPECIES:
            both = (m[f'{s}_B'] > 0) & (m[f'{s}_F'] > 0)
            r = m.loc[both, f'{s}_B'] / m.loc[both, f'{s}_F']
            print(f'  {s:5s} cells in both {int(both.sum()):5d} | only BRAIN {int(((m[f"{s}_B"] > 0) & ~both).sum()):4d} '
                  f'| only NRT {int(((m[f"{s}_F"] > 0) & ~both).sum()):4d} | max |ratio/factor - 1| = '
                  f'{(np.abs(r / FACTOR - 1)).max() if len(r) else np.nan:.2e}')
        CHECKS.append(cell_day_match(CN, 'FINN v1.5 NRT (top days)', dates=nrt_dates))
if CHECKS:
    CHK = pd.DataFrame(CHECKS)
    CHK.to_csv(os.path.join(OUT['tables'], 'processing_checks.csv'), index=False)
    print(CHK.round(4).T.to_string())

## 10. Run record

In [ ]:
import importlib.metadata as im
def ver(p):
    try:
        return im.version(p)
    except Exception:
        return None
META = dict(
    created=time.strftime('%Y-%m-%d %H:%M:%S %Z'), brain_core=bc.__version__, python=platform.python_version(),
    packages={p: ver(p) for p in ['numpy', 'pandas', 'netCDF4', 'pyproj', 'shapely', 'geopandas', 'scipy',
                                   'PseudoNetCDF', 'pseudonetcdf']},
    settings=dict(YEAR=YEAR, EARTH_RADIUS_M=EARTH_RADIUS_M, INTERIOR_MIN_FRACTION=INTERIOR_MIN_FRACTION,
                  CROP=CROP, BLOCK_DAYS=BLOCK_DAYS, N_BOOT=N_BOOT, SEED=SEED, ONLY_DATES=ONLY_DATES),
    inputs_md5=INPUTS, grid=GRID_ATTRS, speciation=SPEC, variables_read=VARS, molar_mass=MM,
    days_processed=len(DAYS_OK), days_missing=list(missing_days.strftime('%Y-%m-%d')),
    time_profile=PROFILE, layer_fractions_default=LAYERS_DEFAULT, layer_profile_relative=LAYER_PROFILE,
    brain_over_finn_input=FACTOR,
    cells_touching_SC=int(len(FSC)), cells_interior=int(FSC.interior.sum()), pm_identities=SPC_CHECK,
    products=PRODUCTS)
bc.save_json(META, os.path.join(OUT['logs'], 'run_metadata.json'))
for d in ['tables', 'figures', 'logs']:
    print(d, sorted(os.listdir(OUT[d])))

## 11. Reading the results

**What the ratios measure.** `R1` ratios compare identical cells, so they contain no region assumption. `R2`
ratios refer to the whole State; the BRAIN side assumes that emissions are spread evenly inside each border cell,
and the low/high bounds show how much that assumption can move the ratio. If R1 and R2 ratios agree, the region
definition does not drive the result.

**What can explain a difference**, in the order the notebook can separate them:

1. *Processing* (section 9): with the default settings BRAIN carries 1.01 × its FINN input. Any other gap
   between BRAIN and the v1.5 input means a different input or setting.
2. *Input file*: BRAIN was built from the near-real-time v1.5 files (inferred from the file names; 9b tests it).
   Fires present only in BRAIN or only in the final annual file (9a) measure this effect.
3. *FINN version* (BRAIN vs FINN v2.5 MODIS): FINN v2.5 uses an updated algorithm to determine fire size
   (Wiedinmyer et al., 2023; Hoinaski et al., 2024, section 2.1.3).
4. *Satellite input* (FINN v2.5 MODIS vs MODIS+VIIRS): in the FINN_SC records of 2019, MODIS+VIIRS holds
   12 334 records in SC and MODIS alone 4 240.

**Statistics.** Monthly correlations use 12 values and are descriptive. Daily correlations come with a
moving-block bootstrap interval because consecutive days are not independent. Cell and municipal correlations
are Spearman (rank) correlations; for municipalities the BRAIN values are areal allocations of 4 km cells, so
small municipalities inherit the emission of shared cells.

**Not compared:** CO₂ (absent from BRAIN), NMOC/COV (split into lumped CB6 species), PM₁₀ (absent).

## References

* Henderson, B. finn2cmaq: fire emission pre-processor for CMAQ. GitHub, https://github.com/barronh/finn2cmaq
  (commits `88bb2f6`, 2022-08-04, and `c3788ce`, 2022-09-16).
* Hoinaski, L., Will, R., Ribeiro, C. B. (2024). Brazilian Atmospheric Inventories – BRAIN: a comprehensive
  database of air quality in Brazil. *Earth System Science Data*, 16, 2385–2405.
  https://doi.org/10.5194/essd-16-2385-2024
* Hoinaski, L., Will, R., Ribeiro, C. B. (2023). Brazilian Atmospheric Inventories – BRAIN version 1: emission
  dataset in Southern Brazil, V1. Science Data Bank. https://doi.org/10.57760/sciencedb.09886
* Künsch, H. R. (1989). The jackknife and the bootstrap for general stationary observations. *The Annals of
  Statistics*, 17(3). https://doi.org/10.1214/aos/1176347265
* PseudoNetCDF 3.5.0 (Python Package Index), functions `ll2ij`, `ij2ll` and the IOAPI map definition.
* Wiedinmyer, C. et al. (2011). The Fire INventory from NCAR (FINN): a high resolution global model to estimate
  the emissions from open burning. *Geoscientific Model Development*, 4, 625–641.
  https://doi.org/10.5194/gmd-4-625-2011
* Wiedinmyer, C. et al. (2023). The Fire Inventory from NCAR version 2.5: an updated global fire emissions model
  for climate and chemistry applications. *Geoscientific Model Development*, 16, 3873–3891.
  https://doi.org/10.5194/gmd-16-3873-2023